**[🏠 Course Home](00_START_HERE.ipynb)** | **📖 [Conceptual](../conceptual/18_case_studies_flaky_tests_and_pipelines.ipynb)** | **🐍 [Python](../python/18_case_studies_flaky_tests_and_pipelines.ipynb)** | **⬅️ [Prev: Ch 17: Tolerance Intervals & Sizing](17_tolerance_intervals_and_decision_sizing.ipynb)** | **➡️ [Next: Ch 19: Verifying Flaky Fixes (SPRT & BF)](19_verifying_flaky_test_fixes.ipynb)**

*🧭 Part V: Production Reliability, CI/CD & Active Learning — Chapter 18 of 22*

---

# Chapter 18: Real-World Case Studies — Flaky Tests, Streaks & Engineering
Pipelines
### *The 50-Sided Die Revisited, The (N, M) Streak Trap, and The End-to-End
Governance Engine*

Throughout Chapters 1 through 16, we constructed a comprehensive Bayesian toolkit:
exact conjugate updating, grid evaluations, quadratic Laplace approximations,
Markov chain sampling, physics-guided Hamiltonian trajectories, dynamic memory
filters with exponential decay, and asymmetric loss optimization.

In this gateway synthesis chapter to Part V (Production Reliability Engineering
and CI/CD Governance), we deploy this entire machinery against the two most
costly reliability dilemmas in modern software engineering:
1. **The Flaky Test Fix Dilemma**: Why $0/100$ passes proves almost nothing,
   how the 50-sided die misleads engineers, and how sequential early-abort
   eliminates $90\%$ of wasted cloud spend.
2. **The Test Promotion Dilemma**: Why heuristic passing streaks (e.g. "passed
   50 times in a row") fall prey to survivorship bias and the gambler's fallacy,
   and how continuous Bayesian decay filters enforce ungameable contracts.
3. **The Master Governance Router**: An automated four-stage pipeline connecting
   Fix Verification, Test Promotion, Suite Tolerance Contracts, and Active
   Changepoint Bisection into an integrated production architecture.

---

## Table of Contents
1. **Part 1: The 50-Sided Die Trap & The 100-Pass Delusion**
   - The $2\%$ Flake Rate as a 50-Sided Die
   - Frequentist Rule of Three vs. Beta-Binomial Posterior
   - Wald's Sequential Probability Ratio Test (SPRT)
2. **Part 2: The (N, M) Passing Streak Trap & Gambler's Fallacy**
   - Survivorship Bias in Staging Gatekeeping
   - Streak Amnesia vs. The Continuous Bayesian Health Meter
   - Simulation: Flaky Tests Sneaking Through Hard Counters
3. **Part 3: End-to-End Reliability Engineering Decision Router**
   - The Four-Pillar Reliability Governance Pipeline
   - Integrated Multi-Panel Synthesis Dashboard
4. **Part 4: Master Reference Matrix & Course Rosetta Stone**
5. **Part 5: Check Your Understanding (3-Tier Scaffolding Exercises)**
   - 🟢 Tier 1 (Warm-Up): The 100-Pass Delusion & Bayes Factor Calculation
   - 🟡 Tier 2 (Guided Calculation): Memory-Decayed Promotion Threshold
   - 🔴 Tier 3 (Production Challenge): Asymmetric Early-Abort Cost Thresholding

---

## Part 1: The 50-Sided Die Trap & Why 0/100 Passes Proves Almost Nothing

### The Scenario
A test fails twice in 100 CI runs on the main branch ($p = 2.0\%$). An engineer
submits a bug fix, executes the test 100 times in a local loop, observes zero
failures ($0/100$), declares the issue resolved, and merges. Two days later,
the test flakes again in release qualification.

### The Mathematics of the 50-Sided Die
A $2\%$ flake rate represents a fair **50-sided die** with 1 red face. The
probability of observing zero red faces in $N = 100$ independent rolls is:
$$P(0 \text{ failures} \mid p = 0.02, N = 100) = (1 - 0.02)^{100}
= 0.98^{100} \approx \mathbf{13.3\%}$$

A $13.3\%$ probability is almost as likely as rolling a 6 on a standard
six-sided gaming die ($16.7\%$). Observing 100 clean passes is weak
circumstantial evidence, not definitive proof of a fix.

### The Frequentist Rule of Three vs. Bayesian Posterior
Under classical statistics, observing zero events in $N$ trials yields a
$95\%$ upper confidence bound governed by the **Rule of Three**:
$$p_{\text{upper}} \approx \frac{3}{N} = \frac{3}{100} = \mathbf{3.0\%}$$
The test began with a $2.0\%$ flake rate; after 100 clean passes, the
frequentist
guaranteed upper bound is $3.0\%$!

Under Bayesian updating with a flat prior $p \sim \text{Beta}(1, 1)$, observing
zero failures in $N$ trials yields the posterior:
$$p \mid D \sim \text{Beta}(1, N + 1)$$
The exact posterior probability that the flake rate exceeds $1\%$ is:
$$P(p > 0.01 \mid D) = (1 - 0.01)^{101} = 0.99^{101} \approx \mathbf{36.2\%}$$
There remains a $36.2\%$ chance that the test still violates a $1\%$ SLA!

In [ ]:
# ==============================================================================
# Part 1: The 50-Sided Die Simulation & Posterior Flake Rate Distribution
# ==============================================================================
par(mfrow = c(1, 2), mar = c(4.5, 4.5, 3.5, 1.5))

n_runs <- 1:200
flake_rates <- c(0.005, 0.01, 0.02, 0.05)
colors <- c("#1f77b4", "#ff7f0e", "#2ca02c", "#d62728")

# 1. Probability of zero failures across trial counts
plot(n_runs, (1 - flake_rates[1])^n_runs, type = "l", col = colors[1], lwd = 2,
     ylim = c(0, 1), xlab = "Number of Test Runs (N)",
     ylab = "Probability of Zero Failures P(0 in N)",
     main = "Probability of Zero Failures (1 - p)^N\n(The 50-Sided Die Trap)")
grid()

for (i in 2:length(flake_rates)) {
  lines(n_runs, (1 - flake_rates[i])^n_runs, col = colors[i], lwd = 2)
}

p_zero_100 <- (1 - 0.02)^100
points(100, p_zero_100, pch = 19, col = "red", cex = 1.4)
abline(v = 100, col = "gray", lty = 2)
abline(h = 0.05, col = "red", lty = 3)

legend("topright", legend = c("p = 0.5%", "p = 1.0%", "p = 2.0%", "p = 5.0%",
                              sprintf("N=100, p=2%% (%.1f%%)", p_zero_100 * 100)),
       col = c(colors, "red"), lwd = c(2, 2, 2, 2, NA), pch = c(NA, NA, NA, NA, 19),
       cex = 0.8)

# 2. Beta-Binomial posterior densities after 0 failures in N runs
theta <- seq(0, 0.10, length.out = 500)
sample_sizes <- c(20, 50, 100, 200)

plot(theta * 100, dbeta(theta, 1, sample_sizes[1] + 1), type = "l",
     col = colors[1], lwd = 2, ylim = c(0, 50),
     xlab = "Flake Rate theta (%)", ylab = "Posterior Density p(theta | D)",
     main = "Posterior Belief After 0 Failures in N Runs\n(Beta(1, N + 1))")
grid()

for (i in 2:length(sample_sizes)) {
  lines(theta * 100, dbeta(theta, 1, sample_sizes[i] + 1), col = colors[i], lwd = 2)
}

for (i in 1:length(sample_sizes)) {
  q95 <- qbeta(0.95, 1, sample_sizes[i] + 1)
  abline(v = q95 * 100, col = colors[i], lty = 2)
}

legend_labels <- sapply(sample_sizes, function(n) {
  q95 <- qbeta(0.95, 1, n + 1)
  sprintf("N = %d (95%% CI < %.1f%%)", n, q95 * 100)
})
legend("topright", legend = legend_labels, col = colors, lwd = 2, lty = 1, cex = 0.8)

par(mfrow = c(1, 1))

q95_100 <- qbeta(0.95, 1, 101)
rule_three_100 <- 3.0 / 100.0
bf_point <- 1.0 / ((1.0 - 0.02)^100)
cat(sprintf("Results for N = 100 clean passes with historical flake rate p = 2.0%%:\n"))
cat(sprintf("  P(0 failures | p = 2.0%%, N = 100)  = %.2f%%\n", p_zero_100 * 100))
cat(sprintf("  Bayesian 95%% Upper Bound on Rate   = %.2f%%\n", q95_100 * 100))
cat(sprintf("  Frequentist Rule of Three (3 / N)  = %.2f%%\n", rule_three_100 * 100))
cat(sprintf("  Likelihood Ratio H_fix / H_unfixed = %.2fx\n", bf_point))

---

## Part 2: The (N, M) Passing Streak Trap & Gambler's Fallacy in CI Gatekeeping

### The Promotion Dilemma
Engineering platforms commonly use heuristic streak criteria to promote test
targets from non-blocking Staging suites to release-blocking presubmit gates:
> *"Promote test $T$ to blocking tier if it has achieved a historical passing
> streak of at least $N = 50$ consecutive runs, and its current streak is at
> least $M = 10$."*

### Why Is This Heuristic Defective?
1. **Survivorship Bias & The Gambler's Fallacy**:
   - For a test with a permanent $p = 2.0\%$ flake rate, the probability of
     passing 50 consecutive runs in any single attempt is $0.98^{50} \approx
     36.4\%$.
   - Across $500$ runs in staging, the probability that the test experiences at
     least one 50-run passing streak purely by chance exceeds **$65\%$**!
   - Streak counters reward lucky clusters of passes while ignoring the
     underlying Poisson failure rate.
2. **Amnesia**:
   - A streak counter looks solely at recent runs and entirely forgets that the
     test failed 4 times immediately prior to the streak.

### The Bayesian Alternative: Continuous Dynamic Memory Filter
Instead of brittle streak counters, Chapter 15 demonstrated how an exponential
memory decay filter ($\\gamma = 0.98$) continuously tracks state:
$$\alpha_t = \gamma \alpha_{t-1} + (1 - y_t), \quad
\beta_t = \gamma \beta_{t-1} + y_t$$

A test is promoted if and only if:
$$P(\theta < 1.0\% \mid \alpha_t, \beta_t) \ge \mathbf{95\%}$$
This rule requires sustained health, cannot be fooled by lucky streaks, and
automatically demotes deteriorating tests without manual intervention.

In [ ]:
# ==============================================================================
# Part 2: The (N, M) Passing Streak Trap vs. Bayesian Dynamic Memory Filter
# ==============================================================================
set.seed(42)

n_total_runs <- 500
true_flake_rate <- 0.02
gamma_run <- 0.98^(1.0 / 20.0)
target_sla <- 0.01

outcomes <- rbinom(n_total_runs, size = 1, prob = true_flake_rate)

current_streak <- integer(n_total_runs)
max_streak_so_far <- integer(n_total_runs)
promoted_streak <- logical(n_total_runs)

streak <- 0
max_streak <- 0
for (t in 1:n_total_runs) {
  if (outcomes[t] == 0) {
    streak <- streak + 1
  } else {
    streak <- 0
  }
  current_streak[t] <- streak
  max_streak <- max(max_streak, streak)
  max_streak_so_far[t] <- max_streak
  if (max_streak >= 50 && streak >= 10) {
    promoted_streak[t] <- TRUE
  }
}

alpha_hist <- numeric(n_total_runs)
beta_hist <- numeric(n_total_runs)
prob_healthy <- numeric(n_total_runs)
posterior_mean <- numeric(n_total_runs)

alpha <- 1.0
beta <- 1.0

for (t in 1:n_total_runs) {
  alpha <- alpha * gamma_run + (1 - outcomes[t])
  beta <- beta * gamma_run + outcomes[t]
  alpha_hist[t] <- alpha
  beta_hist[t] <- beta
  posterior_mean[t] <- beta / (alpha + beta)
  prob_healthy[t] <- pbeta(target_sla, beta, alpha)
}

promoted_bayes <- prob_healthy >= 0.95

par(mfrow = c(2, 1), mar = c(4, 4.5, 3, 1.5))

# Panel 1: Heuristic Streak Trajectory
plot(1:n_total_runs, current_streak, type = "l", col = "#1f77b4",
     ylim = c(0, max(max_streak_so_far) + 10),
     xlab = "", ylab = "Passing Streak (Runs)",
     main = "The (N, M) Passing Streak Trap: Flaky Test (p=2%) Sneaking Through Gates")
grid()
lines(1:n_total_runs, max_streak_so_far, col = "#2ca02c", lwd = 2)
abline(h = 50, col = "red", lty = 2)
legend("topleft", legend = c("Current Streak", "Max Streak", "Threshold N=50"),
       col = c("#1f77b4", "#2ca02c", "red"), lwd = c(1, 2, 1), lty = c(1, 1, 2),
       cex = 0.8)

# Panel 2: Bayesian Dynamic Memory Filter
plot(1:n_total_runs, prob_healthy * 100, type = "l", col = "#2ca02c", lwd = 2,
     ylim = c(0, 100), xlab = "Test Run Index (Timeline)",
     ylab = "Probability / Rate (%)",
     main = "Bayesian Dynamic Memory Filter: Robustly Blocking Promotion")
grid()
lines(1:n_total_runs, posterior_mean * 100, col = "#9467bd", lwd = 2)
abline(h = target_sla * 100, col = "black", lty = 3)
abline(h = 95, col = "green", lty = 2)
legend("topright",
       legend = c("Bayesian Confidence P(theta < 1.0%)", "Posterior Mean Rate (%)",
                  "Target SLA (1.0%)", "Promotion Gate (95%)"),
       col = c("#2ca02c", "#9467bd", "black", "green"),
       lwd = c(2, 2, 1, 1), lty = c(1, 1, 3, 2), cex = 0.8)

par(mfrow = c(1, 1))

cat(sprintf("Summary over %d CI runs (True Flake Rate p = %.1f%%):\n",
            n_total_runs, true_flake_rate * 100))
cat(sprintf("  Maximum streak achieved by chance: %d passes\n", max(max_streak_so_far)))
cat(sprintf("  Runs falsely approved by streak heuristic: %d\n", sum(promoted_streak)))
cat(sprintf("  Runs permitted by Bayesian 95%% gate:       %d\n", sum(promoted_bayes)))

---

## Part 3: End-to-End Reliability Engineering Decision Router

How do modern reliability platforms govern testing pipelines? Rather than
relying on ad-hoc rules, they synthesize the entire Bayesian curriculum into
an automated **Four-Pillar Decision Router**:

```text
               END-TO-END BAYESIAN RELIABILITY GOVERNANCE PIPELINE
               
  [Incoming Patch] 
         │
         ▼
  ┌────────────────────────────────────────────────────────────────────────┐
  │ 1. Fix Verification (Chapter 19 / SPRT)                                 │
  │    Sequential Bayes early-abort: Pass -> -0.019 pts, Fail -> +3.00 pts   │
  │    Saves >90% compute when verifying race condition fixes               │
  └───────────────────────────────────┬────────────────────────────────────┘
                                      │ Pass
                                      ▼
  ┌────────────────────────────────────────────────────────────────────────┐
  │ 2. Test Promotion & Health Meter (Chapter 15 / Chapter 20)                 │
  │    Exponential memory decay (gamma = 0.98) tracking alpha and beta      │
  │    Gate: P(Flake Rate < SLA) >= 95% (Immune to lucky streak traps)      │
  └───────────────────────────────────┬────────────────────────────────────┘
                                      │ Promoted to Suite
                                      ▼
  ┌────────────────────────────────────────────────────────────────────────┐
  │ 3. Suite Tolerance Contracts (Chapters 17 & 21)                        │
  │    Serial Multiplier: P(All Pass) = (1 - p)^K                           │
  │    Allocates SRE error budgets across K tests to prevent suite collapse │
  └───────────────────────────────────┬────────────────────────────────────┘
                                      │ Regression Detected
                                      ▼
  ┌────────────────────────────────────────────────────────────────────────┐
  │ 4. Active Culprit Localization (Chapter 22)                            │
  │    Marginalize unknown rates via Beta conjugacy (O(N) prefix sweep)     │
  │    Active test dispatch via Mutual Information I(Trial; Culprit)        │
  └────────────────────────────────────────────────────────────────────────┘
```

Below, we simulate this complete governance loop and render an integrated
four-panel summary dashboard.

In [ ]:
# ==============================================================================
# Part 3: End-to-End Reliability Engineering Decision Router Dashboard
# ==============================================================================
par(mfrow = c(2, 2), mar = c(4, 4.5, 3, 1.5))

# --- Panel 1: Stage 1 Fix Verification (SPRT Early-Abort) ---
p_broken <- 0.05
p_fixed <- 0.001
log_pass <- log((1 - p_broken) / (1 - p_fixed))
log_fail <- log(p_broken / p_fixed)
b_abort <- log(19.0)
a_accept <- log(1 / 19.0)

sprt_stream <- c(0, 0, 1)
llr_trace <- c(0)
for (outcome in sprt_stream) {
  inc <- ifelse(outcome == 1, log_fail, log_pass)
  llr_trace <- c(llr_trace, llr_trace[length(llr_trace)] + inc)
}

plot(0:(length(llr_trace) - 1), llr_trace, type = "s", col = "#d62728", lwd = 2,
     ylim = c(a_accept - 1, b_abort + 1), xlab = "Run Count",
     ylab = "Log-Likelihood Ratio", main = "Stage 1: Fix Verification (SPRT Early-Abort)")
grid()
abline(h = b_abort, col = "red", lty = 2)
abline(h = a_accept, col = "green", lty = 2)
legend("bottomleft", legend = c("LLR Trace", "Abort (+2.94)", "Accept (-2.94)"),
       col = c("#d62728", "red", "green"), lwd = c(2, 1, 1), lty = c(1, 2, 2), cex = 0.75)

# --- Panel 2: Stage 2 Test Promotion Confidence ---
sample_n <- seq(10, 300, by = 10)
profiles_p <- c(0.001, 0.009, 0.020, 0.040)
colors_prof <- c("#2ca02c", "#1f77b4", "#ff7f0e", "#d62728")

plot(sample_n, pbeta(0.01, 1 + sample_n * profiles_p[1], 1 + sample_n * (1 - profiles_p[1])) * 100,
     type = "l", col = colors_prof[1], lwd = 2, ylim = c(0, 100),
     xlab = "Effective Sample Size (N_eff)", ylab = "Confidence (%)",
     main = "Stage 2: Promotion Confidence P(theta < 1.0%)")
grid()
for (i in 2:4) {
  lines(sample_n, pbeta(0.01, 1 + sample_n * profiles_p[i], 1 + sample_n * (1 - profiles_p[i])) * 100,
        col = colors_prof[i], lwd = 2)
}
abline(h = 95, col = "black", lty = 2)
legend("right", legend = c("0.1%", "0.9%", "2.0%", "4.0%", "Gate (95%)"),
       col = c(colors_prof, "black"), lwd = c(2, 2, 2, 2, 1), lty = c(1, 1, 1, 1, 2), cex = 0.75)

# --- Panel 3: Stage 3 Suite Tolerance Contracts ---
k_suite <- seq(10, 2000, length.out = 100)
test_rates <- c(0.0001, 0.0005, 0.0010, 0.0020)
colors_rates <- c("#2ca02c", "#1f77b4", "#ff7f0e", "#d62728")

plot(k_suite, ((1 - test_rates[1])^k_suite) * 100, type = "l", col = colors_rates[1], lwd = 2,
     ylim = c(0, 100), xlab = "Suite Size (K Tests)", ylab = "Clean Suite Run (%)",
     main = "Stage 3: Suite Availability P(All Pass)")
grid()
for (i in 2:4) {
  lines(k_suite, ((1 - test_rates[i])^k_suite) * 100, col = colors_rates[i], lwd = 2)
}
abline(h = 99, col = "green", lty = 3)
legend("topright", legend = c("p=0.01%", "p=0.05%", "p=0.10%", "p=0.20%", "99% Budget"),
       col = c(colors_rates, "green"), lwd = c(2, 2, 2, 2, 1), lty = c(1, 1, 1, 1, 3), cex = 0.75)

# --- Panel 4: Stage 4 Active Culprit Localization ---
post_probs <- c(5.5, 9.5, 65.2, 19.9)
barplot(post_probs, names.arg = c("c0", "c1", "c2", "c3"),
        col = c("#aec7e8", "#aec7e8", "#1f77b4", "#aec7e8"),
        ylim = c(0, 80), xlab = "Candidate Revisions",
        ylab = "Posterior Probability (%)", main = "Stage 4: Culprit Posterior P(C = c | D)")
grid()

par(mfrow = c(1, 1))

---

## Part 4: Master Reference Matrix & Course Rosetta Stone

The table below summarizes the entire conceptual and mathematical landscape
developed across this course:

| Sheet / Module | Core Intuition | Mathematical Engine | Production Use Case | Failure Mode / Boundary |
| :--- | :--- | :--- | :--- | :--- |
| **Chapter 1: Foundations** | Balance scale weighing evidence | Beta-Binomial conjugacy ($O(1)$ updating) | Real-time conversion & pass-rate tracking | Restricted to conjugate likelihood pairs |
| **Chapter 5: Grid Approximations** | Cookie sheet buckets | Discretized numerical integration | Low-dimensional non-standard priors ($D \le 3$) | Curse of dimensionality ($O(K^D)$) |
| **Chapter 6: Laplace / QUAP** | Mountain peak curvature | Second Taylor expansion & Hessian $\mathbf{H}^{-1}$ | High-speed prototyping & large $N$ | Distorts skewed tails & multimodal peaks |
| **Chapter 9: Metropolis MCMC** | King Markov exploring islands | Acceptance ratio $\min(1, \alpha)$ | General posterior sampling from scratch | Random walk diffusion in high dimensions |
| **Chapter 10: Hamiltonian MC** | Frictionless rollercoaster | Hamilton's equations & leapfrog steps | Production Bayesian models (Stan / PyMC) | Divergent transitions on narrow ridges |
| **Chapter 15: Dynamic Filtering** | Fading ink & memory decay | Exponential discount $\gamma$ ($O(1)$ memory) | Streaming telemetry & moving target tracking | Calibration of memory half-life parameter |
| **Chapter 16: Decision Theory** | Asymmetric smoke alarm loss | Minimizing posterior loss $\mathbb{E}[L(a, \theta)]$ | Setting optimal business thresholds ($c^*$) | Requires accurate cost elicitation |
| **Chapter 18 & Appendices** | Automated reliability governance | SPRT early-abort, tolerance contracts & entropy | CI/CD gatekeeping & culprit bisect search | Flake noise masking subtle regressions |

---

## 🎯 Check Your Understanding: 3-Tier Scaffolding Exercises

Sharpen your intuition and practice the mathematical concepts introduced in this
sheet. Write your solutions in the interactive attempt cells below before
expanding the verified solutions.

---

### 🟢 Tier 1 (Warm-Up): The 100-Pass Delusion & Bayes Factor Calculation
> **Scenario**: A flaky test with historical flake rate $p_0 = 3.0\%$ has a code
> patch applied. You run $N = 100$ local runs and observe zero failures ($k =
0$).
>
> 1. What is the frequentist $95\%$ upper bound on the failure rate via the
>    Rule of Three?
> 2. Under a flat prior $p \sim \text{Beta}(1, 1)$, what is the exact Bayesian
>    $95\%$ posterior credible upper bound?
> 3. What is the point likelihood ratio $BF_{10} = \frac{P(D \mid p = 0)}{P(D
\mid p = 0.03)}$?

In [ ]:
# ============================================================
# 🟢 Tier 1: Your Turn — Attempt the Problem Here
# ============================================================
n <- 100
p_historical <- 0.03

# 1. Rule of Three upper bound:
# rule_of_three <- ...

# 2. Bayesian 95% credible upper bound under Beta(1, n + 1):
# bayes_upper <- ...

# 3. Likelihood ratio BF_10 (perfect fix p=0 vs unfixed p=0.03):
# bf_10 <- ...

<details>
<summary><b>🔍 Click here to view the solution & mathematical
derivation</b></summary>

**Answers**:
1. **Rule of Three**: $p_{\text{upper}} = \frac{3}{100} = \mathbf{3.0\%}$.
2. **Bayesian $95\%$ Credible Bound**: $q_{0.95} = 1 - (1 - 0.95)^{1/101}
   \approx \mathbf{2.92\%}$.
3. **Likelihood Ratio**: $BF_{10} = \frac{1.0^{100}}{(1 - 0.03)^{100}}
   = \frac{1.0}{0.04755} \approx \mathbf{21.03}$.

**Derivation**:
Observing 100 passes provides a 21-fold likelihood boost in favor of a fix over
a $3\%$ flake rate, but your $95\%$ credible bound is still $2.92\%$, barely
lower than the starting point!
</details>

---

### 🟡 Tier 2 (Guided Calculation): Memory-Decayed Promotion Threshold
> **Scenario**: A CI test suite runs 20 times per day with daily memory decay
> $\gamma = 0.95$. In steady state, the effective sample size is:
> $$N_{\text{eff}} = \frac{1}{1 - \gamma} \times 20 = 400 \text{ runs}$$
> A candidate test has an underlying true flake rate $p = 0.8\%$. The team's
> promotion gate requires $P(\theta < 1.0\% \mid \text{Data}) \ge 95\%$.
>
> Calculate the posterior parameters $(\alpha, \beta)$ and evaluate whether
> the test is promoted.

In [ ]:
# ============================================================
# 🟡 Tier 2: Your Turn — Attempt the Problem Here
# ============================================================
gamma <- 0.95
runs_per_day <- 20
p_true <- 0.008
target_sla <- 0.01

# Calculate steady-state effective sample size:
# n_eff <- ...
# alpha <- ...
# beta <- ...
# prob_healthy <- pbeta(target_sla, beta, alpha)

<details>
<summary><b>🔍 Click here to view the solution & analysis</b></summary>

**Answer**: $P(\theta < 1.0\%) \approx \mathbf{52.9\%}$. **Promotion is
BLOCKED.**

**Derivation**:
1. $N_{\text{eff}} = \frac{20}{1 - 0.95} = 400$ runs.
2. Expected failures $\beta = 1 + 400 \times 0.008 = 4.2$.
3. Expected passes $\alpha = 1 + 400 \times (1 - 0.008) = 397.8$.
4. Under $\text{Beta}(4.2, 397.8)$, the cumulative probability below $0.01$ is
   approximately $52.9\%$. Even though the expected rate ($0.8\%$) is below
   the SLA ($1.0\%$), parameter uncertainty with $N_{\text{eff}} = 400$ is too
   wide to guarantee $95\%$ confidence!
</details>

---

### 🔴 Tier 3 (Production Challenge): Asymmetric Early-Abort Thresholding
> **Scenario**: In a high-throughput pipeline:
> - Test execution cost: $C_{\text{run}} = \$0.50$ per run.
> - Outage cost from releasing a flaky bug: $C_{\text{FN}} = \$20{,}000$.
> - Wasted triage cost from falsely rejecting a good fix: $C_{\text{FP}} =
\$500$.
>
> 1. Derive the calibrated error probabilities $\alpha =
\frac{C_{\text{run}}}{C_{\text{FP}}}$
>    and $\beta = \frac{C_{\text{run}}}{C_{\text{FN}}}$.
> 2. Compute Wald's SPRT stopping boundaries $A$ (accept) and $B$ (abort).

In [ ]:
# ============================================================
# 🔴 Tier 3: Your Turn — Attempt the Problem Here
# ============================================================
c_run <- 0.50
c_fp <- 500.0
c_fn <- 20000.0

# 1. Calibrate error rates:
# alpha_err <- ...
# beta_err <- ...

# 2. Wald SPRT log-boundaries:
# b_upper <- ... # log((1 - beta_err) / alpha_err)
# a_lower <- ... # log(beta_err / (1 - alpha_err))

<details>
<summary><b>🔍 Click here to view the solution & proof</b></summary>

**Answers**:
1. $\alpha = \frac{0.50}{500} = \mathbf{0.001}$ ($0.1\%$ false abort rate).  
$\beta = \frac{0.50}{20{,}000} = \mathbf{0.000025}$ ($0.0025\%$ false pass
rate).
2. Upper Abort Boundary:
$$B = \ln\left(\frac{1 - \beta}{\alpha}\right) =
\ln\left(\frac{0.999975}{0.001}\right)
   \approx \mathbf{+6.91}$$
   Lower Accept Boundary:
$$A = \ln\left(\frac{\beta}{1 - \alpha}\right) =
\ln\left(\frac{0.000025}{0.999}\right)
   \approx \mathbf{-10.60}$$

Because missed outages are 40 times more costly than false alarms, the accept
boundary requires much deeper evidence than the abort boundary.
</details>

---

## 🎓 Conclusion: The Bayesian Superpower in Production

You have now completed the core foundational and case study curriculum of
Bayesian Inference.

You understand:
* Why single-number point estimates fail under small samples and asymmetric
loss.
* Why 100 clean passes is merely a roll of a 50-sided die.
* How dynamic memory decay eliminates the gambler's fallacy in streak gates.
* How to combine information theory, changepoint sweeps, and sequential testing
  into an automated reliability governance platform.

---

8](../conceptual/18_case_studies_flaky_tests_and_pipelines.ipynb) | ↩️
Previous: [Chapter 16: Decision
Theory](16_bayesian_decision_theory_and_predictive_checks.ipynb)**

> 📚 **Deepen Your Engineering Specialization in the Appendices**:
> - **[Chapter 7: The Hessian
Matrix](07_understanding_the_hessian_matrix.ipynb)**
> - **[Chapter 17: Tolerance
Intervals](17_tolerance_intervals_and_decision_sizing.ipynb)**
> - **[Chapter 19: Flaky Test
Verification](19_verifying_flaky_test_fixes.ipynb)**
> - **[Chapter 20: Test
Promotion](20_test_promotion_heuristics_vs_bayes.ipynb)**
> - **[Chapter 21: Framework
Reliability](21_framework_reliability_and_tolerance_contracts.ipynb)**
> - **[Chapter 22: Active Culprit
Search](22_information_theory_and_culprit_search.ipynb)**

---
## 🧭 Course Navigation

* 🏠 **[Course Home & Master Syllabus](00_START_HERE.ipynb)**
* ⬅️ **Previous**: **[Ch 17: Tolerance Intervals & Sizing](17_tolerance_intervals_and_decision_sizing.ipynb)** *(Part IV: Dynamic Systems, Decision Theory & Predictive Guarantees)*
* ➡️ **Next**: **[Ch 19: Verifying Flaky Fixes (SPRT & BF)](19_verifying_flaky_test_fixes.ipynb)** *(Part V: Production Reliability, CI/CD & Active Learning)*
* 🔄 **Companion Tracks for Chapter 18**: [📖 Conceptual](../conceptual/18_case_studies_flaky_tests_and_pipelines.ipynb) | [🐍 Python](../python/18_case_studies_flaky_tests_and_pipelines.ipynb) | [🔵 R](../r/18_case_studies_flaky_tests_and_pipelines.ipynb)
